# 🧪 Lab Analysis Pipeline: UV-Vis & Granulometer Toolkit
### Complete Tutorial, Workflow & Technical Reference Notebook

This Jupyter Notebook provides a comprehensive guide to using the `lab_analysis` package for:
1. **UV-Vis Spectroscopy Analysis** (`lab_analysis.uvvis`): Pigment quantification (Chlorophylls $a$, $b$, Total Chl using Ritchie 2008 ethanol equations), quality control checks (turbidity at $750\text{ nm}$, carotenoid ratio at $520\text{ nm}$, absorbance saturation $A > 1.0$), and spectral plotting with replicate shading.
2. **Laser-Diffraction Granulometry** (`lab_analysis.granulometer`): Particle size distributions ($q(\%)$, cumulative Passant $\%$), volumetric metrics ($D_{10}, D_{50}, D_{90}$, Span, Variance), and comparative analysis across sonication time, sample volume, or sample types.

---

## 📁 1. Project Directory & Architecture Structure

```text
Internship_Anthony_Data_Processing/
├── pyproject.toml               # Package metadata & entry points
├── requirements.txt             # Core dependencies (numpy, matplotlib, pandas, openpyxl)
├── data/
│   └── raw/                     # Input spectral (.txt, .csv) & particle size files
├── outputs/                     # Generated PNG figures, summary CSVs, & run info JSON
└── src/
    └── lab_analysis/            # Core package
        ├── __init__.py
        ├── common/              # Shared primitives across methods
        │   ├── io.py            # Encoding-agnostic readers & file helpers
        │   ├── plotting.py      # Project-wide matplotlib style & export
        │   └── reports.py       # UTF-8-BOM CSV export helpers
        ├── granulometer/        # Laser diffraction particle-size pipeline
        │   ├── config.py        # Granulometer settings & color palettes
        │   ├── grouping.py      # Replicate & condition grouping
        │   ├── models.py        # Measurement & Condition dataclasses
        │   ├── parsing.py       # Data parsers (.txt, .xlsx)
        │   ├── plotting.py      # Distribution curves & comparison renders
        │   ├── reports.py       # Summary CSV generators
        │   ├── runner.py        # Pipeline orchestrator
        │   └── stats.py         # Particle size metrics (D10, D50, D90, Span)
        └── uvvis/               # UV-Vis spectroscopy & pigment pipeline
            ├── config.py        # Target wavelengths, thresholds, & palettes
            ├── models.py        # Spectrum & Condition dataclasses
            ├── parsing.py       # Spectrum file reader (.txt, .csv)
            ├── plotting.py      # Spectral curves & vertical markers
            ├── reports.py       # Pigment & absorbance CSV summaries
            ├── runner.py        # UV-Vis pipeline orchestrator
            └── stats.py         # Ritchie 2008 chlorophyll equations & QC flags
```


## ⚙️ 2. Environment Setup & Package Installation

To install the package in editable mode from the repository root:

In [ ]:
# Shell command to install package in editable mode
!pip install -e .

In [ ]:
import sys
from pathlib import Path

# Ensure src is in Python path for notebook execution
sys.path.insert(0, str(Path("src").resolve()))

import lab_analysis
print(f"Loaded lab_analysis version: {lab_analysis.__version__}")

---
## 🌈 3. UV-Vis Spectroscopy & Pigment Quantification (`uvvis`)

### Core Features:
- **Target Wavelengths (Ethanol)**: $362, 470, 520, 632, 649, 664, 695, 750\text{ nm}$
- **Ritchie (2008) Ethanol Equations**:
  $$C_{\text{Chl } a} (\mu\text{g/mL}) = 13.36 \cdot A_{664} - 5.19 \cdot A_{649}$$
  $$C_{\text{Chl } b} (\mu\text{g/mL}) = 27.43 \cdot A_{649} - 8.12 \cdot A_{664}$$
  $$\text{Total Chl } (\mu\text{g/mL}) = C_{\text{Chl } a} + C_{\text{Chl } b}$$
- **Specific Extraction Normalization**:
  $$\text{Chl (}\mu\text{g/mg d'extrait)} = \frac{C (\mu\text{g/mL}) \times \text{Facteur de dilution} \times V_{\text{EtOH}} (\text{mL})}{\text{Dry Mass Total (mg)}}$$
- **Quality Control Checks**:
  1. **Turbidity Check**: Flags if $A_{750} > 0.02$ (should equal ~0).
  2. **Carotenoid Ratio Check**: Flags if $A_{520} / A_{\text{RedMax}} \ge 10\%$.
  3. **Linearity Check**: Flags if $A_{\max} > 1.0$ (indicates saturation; re-dilution recommended).


### 3.1 Customizing UV-Vis Configuration Programmatically

In [ ]:
from lab_analysis.uvvis import config as uvvis_config

# Adjust sample extraction parameters
uvvis_config.DEFAULT_DILUTION_FACTOR = 1.0     # Dilution factor
uvvis_config.DEFAULT_EXTRACT_VOLUME_ML = 3.0   # V d'éthanol de reprise (mL)
uvvis_config.DEFAULT_DRY_MASS_MG = 10.0        # Dry mass total (mg)

# Quality thresholds
uvvis_config.MAX_ABSORBANCE_WARNING = 1.0
uvvis_config.TURBIDITY_THRESHOLD_A750 = 0.02
uvvis_config.CAROTENOID_MAX_RATIO_A520 = 0.10

# Display options
uvvis_config.SHOW_WAVELENGTH_MARKERS = True    # Vertical dashed lines at 470, 520, 649, 664 nm
uvvis_config.SHOW_SD_BAND = True               # Color shading across replicates


### 3.2 Executing the Full UV-Vis Pipeline

In [ ]:
from lab_analysis.uvvis import run as run_uvvis

# Run automated UV-Vis analysis on a dataset folder
# Accepts folders containing .txt or .csv spectrum exports
spectra, conditions = run_uvvis(
    input_dir="data/raw/2026_10_30_SF_amplitude",
    output_dir="outputs/uvvis_results"
)

print(f"Processed {len(spectra)} individual spectra across {len(conditions)} sample conditions.")

### 3.3 Inspecting Calculated Pigment Results & Quality Flags

In [ ]:
for spec in spectra[:3]:
    print(f"--- Sample: {spec.name} ---")
    print(f"  A664: {spec.stats.get('A_664'):.4f} | A649: {spec.stats.get('A_649'):.4f} | A750: {spec.stats.get('A_750'):.4f}")
    print(f"  Chl a: {spec.stats.get('Chl_a_ug_mL'):.3f} µg/mL  -->  {spec.stats.get('Chl_a_ug_mg'):.3f} µg/mg extract")
    print(f"  Chl b: {spec.stats.get('Chl_b_ug_mL'):.3f} µg/mL  -->  {spec.stats.get('Chl_b_ug_mg'):.3f} µg/mg extract")
    print(f"  Total Chl: {spec.stats.get('Total_Chl_ug_mg'):.3f} µg/mg extract")
    if spec.flags:
        print(f"  ⚠️ QC Flags: {spec.flags}")
    else:
        print("  ✅ QC Status: Passed all criteria.")

---
## 📏 4. Laser Diffraction Granulometry (`granulometer`)

### Core Features:
- **Distribution Analysis**: Differential $q(\%)$ and cumulative Passant $(\%)$.
- **Particle Size Statistics**: $D_{10}, D_{50}, D_{90}$, Span $=\frac{D_{90} - D_{10}}{D_{50}}$, Mean, SD, Variance.
- **Color Palettes**: Replicate shading using custom palettes (`ocean`, `teal`, `amber`, `violet`, `rose`, `green`).
- **Comparative Analysis**: Compare across sonication time, volume, or sample type.


### 4.1 Customizing Granulometer Configuration

In [ ]:
from lab_analysis.granulometer import config as gran_config

# Set plot outputs
gran_config.PLOT_KINDS = ("combined",)      # "combined" (2-panel), "q", "passant", "both"
gran_config.COMPARE_OVER = "time"           # Compare over "time", "volume", or "sample"
gran_config.SHOW_SD_BAND = True            # Shading for replicate standard deviation
gran_config.WARN_CV_D50_PERCENT = 10.0     # Flag conditions where D50 CV > 10%

# Color Palette selection
print("Available palettes:", list(gran_config.PALETTES.keys()))
gran_config.GROUP_PALETTES = {"SF": "teal"} # Pin sample "SF" to the teal palette

### 4.2 Executing the Granulometer Pipeline

In [ ]:
from lab_analysis.granulometer import run as run_granulometer

# Run automated particle-size analysis
measurements, conditions = run_granulometer(
    input_dir="data/raw/2026_10_26_SF_test",
    output_dir="outputs/granulometer_results"
)

print(f"Processed {len(measurements)} measurements into {len(conditions)} conditions.")

---
## 🖥️ 5. Command-Line Interface (CLI) Usage

Both subpackages expose direct terminal commands:

### UV-Vis Spectroscopy CLI:
```bash
# Run analysis with default settings
uvvis-analysis data/raw/2026_10_30_SF_amplitude --output-dir outputs/uvvis_cli_run

# Specify solvent explicitly
uvvis-analysis data/raw/2026_10_30_SF_amplitude --solvent ethanol
```

### Granulometer CLI:
```bash
# Basic run
granulometer-analysis data/raw/2026_10_26_SF_test --output-dir outputs/gran_cli_run

# Custom comparative run over sonication time
granulometer-analysis data/raw/2026_10_26_SF_test --compare-over time --plot-kinds combined
```


---
## 📊 6. Output Files Summary

Running either pipeline generates the following output structure:

```text
outputs/
├── individual/            # Individual spectrum/particle size distribution plots (.png)
├── replicates/            # Replicate overlay plots with color shading and SD bands (.png)
├── comparisons/           # Comparison figures by condition/time/volume (.png)
├── overview/              # Multi-panel overview grid of all conditions (.png)
└── summary/
    ├── spectrum_summary.csv    # Individual spectral readings, Chl concentrations & QC flags
    ├── condition_summary.csv   # Mean ± SD for replicates per condition
    └── _run_info.json          # Execution metadata, parameters & timestamps
```
